In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

import brain_probability_flux.plotting as bpp
from brain_probability_flux.dina.dynamics import ising as idyna

In [ ]:
# Results are read from data/. To plot a fresh run, set DATA_DIR to its run
# directory, for example Path("../data/2026-10-20").
DATA_DIR = bpp.DATA_DIR

In [ ]:
max_fig_width = bpp.FIG_WIDTH
max_fig_height = bpp.FIG_HEIGHT

In [ ]:
pfa, cluster_inds = bpp.aggregated_pfa(data_dir=DATA_DIR)
num_clusters = len(np.unique(cluster_inds))
color = bpp.spin_colors(num_clusters)
pfa.perform_WPCA(spin_colors=color)

In [ ]:
flux_matrices = bpp.flux_matrices_by_task(pfa)
transition_indices = bpp.get_one_bit_difference_indices(num_clusters)
inferred = bpp.load_results("inferred_networks", "arrhenius", DATA_DIR)

In [ ]:
# Hypercube states annotated with a state glyph, and where the glyphs sit
inset_states = np.array(
    [
        [-1, -1, 1, 1, 1, -1, 1],
        [-1, 1, 1, 1, 1, -1, 1],
    ]
)
annot_coords = np.array(
    [
        [2.45, 0.9],
        [2.3, 1.8],
    ]
)

# Fig. 6

In [ ]:
# Flux of the inferred model of each task, and its correlation with the
# observed flux
model_flux = [
    idyna.im_to_flux(
        inferred["beta_J"][k],
        1.0,
        magnetic_field=inferred["beta_h"][k],
        rate_const=inferred["rate_const"][k],
    )
    for k in range(bpp.NUM_TASKS)
]
r_values = [
    stats.pearsonr(
        flux_matrices[k][transition_indices], model_flux[k][transition_indices]
    ).statistic
    for k in range(bpp.NUM_TASKS)
]

fig, axs = plt.subplots(
    3, 3, figsize=(max_fig_width, max_fig_height), sharex=True, sharey=True
)
bpp.plot_pca_by_task(
    pfa,
    axs,
    color=color,
    arrows=[idyna.flux_to_flux_list(flux) for flux in model_flux],
    titles=[
        f"{name}, $r={r:.2f}$" for name, r in zip(pfa.task_name_list, r_values)
    ],
)
for ax in bpp.task_axes(axs):
    for state, annot_coord in zip(inset_states, annot_coords):
        bpp.state_indicator(ax, state, annot_coord, pfa.wpca.components_)
bpp.save_panel(fig, "../figures/fig-06", "fig-06", bbox_inches=None)